In [ ]:
from roundel_clinical_utils import *

data_path = '/workspaces/Roundel-Clinical/roundel/data'
os.environ["CUDA_VISIBLE_DEVICES"]="0"

In [ ]:
patient = '1.3.6.1.4.1.53684.1.1.3.1671342546.2056.1623765873.123561'
image = load_nii(f'{data_path}/image___{patient}.nii.gz')
mask = load_nii(f'{data_path}/masks___{patient}.nii.gz')

In [ ]:
ps = 1.172
thickness = 8.0

In [ ]:
image = zoom(image, (ps,ps,1,1), order = 1)

In [ ]:
model = tf.keras.models.load_model('UNet3plus.h5', 
                                   compile = False,
                                   custom_objects={"InstanceNormalization":InstanceNormalization,
                                                   "ResizeAndConcatenate":ResizeAndConcatenate})

In [ ]:
def segment_image(image):
    # Crop and pad the image to correct shape
    target_shape = (256,256)
    mask = []
    for t in range(image.shape[-1]):  
        image_cropped, meta =  crop_pad_image_only(image[..., t], target_shape = target_shape)
        print(meta)
        X = z_normalise_image(image_cropped)[np.newaxis,...,np.newaxis]

        print(image_cropped.shape)
        pred_mask = sliding_window_inference_3d(
                        model,
                        X,                   # np.ndarray [batch_size,Y,X,Z,1] already preprocessed & normalised
                        patch_size=[256,256,10],
                        overlap=0.5,
                        apply_softmax=False,       # set False as the model already uses softmax activation
                        out_channels=5,    # if known, can be set to avoid dry run
                        tta=False,                 # Whether to use test-time augmentation (flips)
                        plot_tta=False,           # Whether to plot the prediction after each TTA variant (for debugging)
                        scan_id = None,           # used for naming the TTA variant plots
                        time_step_counter=0, # used for naming the TTA variant plots
                        gaussian_sigma_scale=1/8, # controls how peaked the Gaussian is
                        deep_supervision=True,
                        run = None               # neptune run instance for logging
                    )
        print(pred_mask.shape)
        pred_mask_one_hot = reverse_crop_pad(pred_mask, meta)
        pred_mask_one_hot = get_one_hot(pred_mask_one_hot.astype(np.uint8), 5)  # [Y0,X0,Z0,C_out]

        mask.append(pred_mask_one_hot)
    mask = np.stack(mask, -2)
    return mask

In [ ]:
mask = segment_image(image)

In [ ]:
t = 5
s = 5
c = 1
plt.imshow(image[...,s,t], 'gray')
plt.imshow(mask[...,s, t, c], alpha = mask[...,s, t, c]/2)